In [15]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

text_loader = TextLoader(
    file_path="data/split_text.txt",
    encoding="utf-8"
)

# load()의 결과는 [Document(...)] 형태의 '리스트'입니다.
doc = text_loader.load()

# ==========================================
# [상황 1] 질문자님이 겪으신 원본 설정
# ==========================================
splitter_1 = CharacterTextSplitter(
    separator='\n', # 💡 문제점: 줄바꿈 기호가 나올 때만 자르겠다는 뜻입니다.
    chunk_size=30,  # 30자 단위로 자르고 싶지만...
    chunk_overlap=20
)

chunks_1 = splitter_1.split_text(doc[0].page_content)
# [결과] -> ['머신러닝은 데이터로부터 스스로 학습하는 기술입니다.']
#
# [이유 주석]:
# 문장 전체(30자)가 끝날 때까지 줄바꿈('\n')이 단 한 번도 나오지 않았습니다.
# 스플리터는 글자 중간을 무작위로 찢을 수 없으므로, 줄바꿈이 나올 때까지
# 무조건 한 덩어리로 유지합니다. (실제 긴 문장에서는 30자를 초과했다며 경고를 띄웁니다.)
# 한 덩어리로 끝나버렸기 때문에 앞뒤로 겹칠 다른 조각이 없어서 겹치기(overlap)도 작동하지 않습니다.


# ==========================================
# [상황 2] separator=' ', size=10, overlap=5 설정
# ==========================================
splitter_2 = CharacterTextSplitter(
    separator=' ',  # 💡 해결책: 이제 띄어쓰기(공백)를 만날 때마다 자를 준비를 합니다.
    chunk_size=10,  # 최대 10글자 근처에서 자릅니다.
    chunk_overlap=5 # 다음 조각을 만들 때 앞 조각의 마지막 5글자만큼 겹쳐서 가져옵니다.
)

chunks_2 = splitter_2.split_text(doc[0].page_content)
# [결과] -> ['머신러닝은 데이터로부터', '데이터로부터 스스로', '스스로 학습하는', '학습하는 기술입니다.']
#
# [이유 주석]:
# 1. 첫 조각을 10자 근처인 "머신러닝은 데이터로부터" (12자)까지 공백 기준으로 담습니다.
# 2. 다음 조각을 만들 때, 겹치기(overlap=5) 설정 때문에 앞 조각의 뒷부분인 "데이터로부터"를 기억합니다.
# 3. 그래서 두 번째 조각은 앞부분과 겹친 "데이터로부터 스스로"가 됩니다.
# 4. 세 번째 조각 역시 앞 조각의 뒷부분인 "스스로"를 겹치면서 "스스로 학습하는"이 됩니다.
#
# 이처럼 구분자를 공백(' ')으로 주어야만 비로소 chunk_size와 overlap이 의미 있게 작동합니다

print(chunks_1)
print(chunks_2)

Created a chunk of size 51, which is longer than the specified 30
Created a chunk of size 45, which is longer than the specified 30
Created a chunk of size 11, which is longer than the specified 10


['머신러닝은 데이터로부터 스스로 학습하는 기술입니다. 지도 학습은 레이블 데이터를 사용합니다.', '비지도 학습은 레이블 없이 구조를 찾습니다. 강화 학습은 보상을 통해 학습합니다.', '딥러닝은 신경망을 여러 층으로 쌓아 학습합니다. 자연어 처리는 텍스트를 이해하는 기술입니다.']
['머신러닝은', '데이터로부터 스스로', '스스로 학습하는', '기술입니다. 지도', '지도 학습은 레이블', '레이블 데이터를', '사용합니다.\n비지도', '학습은 레이블 없이', '없이 구조를', '구조를 찾습니다.', '찾습니다. 강화', '강화 학습은 보상을', '보상을 통해', '학습합니다.\n딥러닝은', '신경망을 여러', '여러 층으로 쌓아', '쌓아 학습합니다.', '자연어 처리는', '처리는 텍스트를', '텍스트를 이해하는', '기술입니다.']
